# Cross-Dataset Evaluation on Kvasir-Sessile

Evaluates trained Kvasir-SEG models on Kvasir-Sessile with harmonized metadata and transfer-gap reporting.


In [1]:
import sys
from pathlib import Path

def _bootstrap_kvasir_seg_path() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / 'utils' / 'segmentation_common.py').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
        alt = p / 'Prototyping_reformat' / 'DatasetAnalysis' / 'Kvasir_SEG'
        if (alt / 'utils' / 'segmentation_common.py').exists():
            if str(alt) not in sys.path:
                sys.path.insert(0, str(alt))
            return alt
    raise RuntimeError('Could not locate Kvasir_SEG utils path from current working directory.')

BOOTSTRAP_ROOT = _bootstrap_kvasir_seg_path()
print('BOOTSTRAP_ROOT:', BOOTSTRAP_ROOT)


BOOTSTRAP_ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG


In [2]:
import json
import os
from pathlib import Path

import pandas as pd

from utils.segmentation_common import (
    build_kvasir_sessile_metadata,
    ensure_kvasir_sessile_data,
    evaluate_run_dir_on_metadata,
    find_kvasir_seg_root,
)

ROOT = find_kvasir_seg_root()
OUT_DIR = ROOT / '3_generalization_and_ablation' / 'out' / 'cross_dataset_kvasir_sessile'
OUT_DIR.mkdir(parents=True, exist_ok=True)

ALLOW_DOWNLOAD = os.getenv('ALLOW_DOWNLOAD', '0') == '1'
ALLOW_HF_DOWNLOAD = os.getenv('ALLOW_HF_DOWNLOAD', '0') == '1'
COMPUTE_HD95 = os.getenv('COMPUTE_HD95', '0') == '1'

MAX_EXTERNAL_SAMPLES = int(os.getenv('MAX_EXTERNAL_SAMPLES', '0')) or None
EVAL_THRESHOLD = os.getenv('EVAL_THRESHOLD', '').strip()
EVAL_THRESHOLD = float(EVAL_THRESHOLD) if EVAL_THRESHOLD else None
EVAL_BATCH_SIZE = int(os.getenv('EVAL_BATCH_SIZE', '0')) or None
EVAL_NUM_WORKERS = os.getenv('EVAL_NUM_WORKERS', '').strip()
EVAL_NUM_WORKERS = int(EVAL_NUM_WORKERS) if EVAL_NUM_WORKERS else None
EVAL_IMAGE_SIZE = int(os.getenv('EVAL_IMAGE_SIZE', '0')) or None

RUN_FILTER = [x.strip() for x in os.getenv('EVAL_RUN_FILTER', '').split(',') if x.strip()]

print('ROOT:', ROOT)
print('OUT_DIR:', OUT_DIR)
print('ALLOW_DOWNLOAD:', ALLOW_DOWNLOAD)
print('ALLOW_HF_DOWNLOAD:', ALLOW_HF_DOWNLOAD)
print('COMPUTE_HD95:', COMPUTE_HD95)
print('MAX_EXTERNAL_SAMPLES:', MAX_EXTERNAL_SAMPLES)
print('RUN_FILTER:', RUN_FILTER if RUN_FILTER else '<all>')


ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG
OUT_DIR: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/cross_dataset_kvasir_sessile
ALLOW_DOWNLOAD: False
ALLOW_HF_DOWNLOAD: False
COMPUTE_HD95: False
MAX_EXTERNAL_SAMPLES: None
RUN_FILTER: <all>


In [3]:
sessile_dir = ensure_kvasir_sessile_data(ROOT, allow_download=ALLOW_DOWNLOAD)
print('Kvasir-Sessile dir:', sessile_dir)

sessile_df = build_kvasir_sessile_metadata(sessile_dir, split_name='external_test')

if MAX_EXTERNAL_SAMPLES:
    sessile_df = sessile_df.sample(n=min(MAX_EXTERNAL_SAMPLES, len(sessile_df)), random_state=42).reset_index(drop=True)

meta_csv = OUT_DIR / 'sessile_metadata_external_test.csv'
sessile_df.to_csv(meta_csv, index=False)

print('Samples:', len(sessile_df))
print('Saved metadata:', meta_csv)
display(sessile_df.head())


Kvasir-Sessile dir: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/out/raw/kvasir_sessile/sessile-main-Kvasir-SEG
Samples: 196
Saved metadata: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/cross_dataset_kvasir_sessile/sessile_metadata_external_test.csv


,img_id,image_path,mask_path,width,height,fg_pixels,total_pixels,mask_area_ratio,component_count,bbox_count,split,split_seed,source_dataset
0,cju0qoxqj9q6s0835b43399p4,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,1348,1070,390295,1442360,0.270595,1,0,external_test,-1,kvasir_sessile
1,cju0sxqiclckk08551ycbwhno,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,570,531,13168,302670,0.043506,1,0,external_test,-1,kvasir_sessile
2,cju0tl3uz8blh0993wxvn7ly3,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,622,529,1735,329038,0.005273,1,0,external_test,-1,kvasir_sessile
3,cju13hp5rnbjx0835bf0jowgx,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,622,531,35189,330282,0.106542,1,0,external_test,-1,kvasir_sessile
4,cju14g8o4xui30878gkgbrvqj,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,622,531,8522,330282,0.025802,1,0,external_test,-1,kvasir_sessile


In [4]:
search_roots = [
    ROOT / '1_classic_seg_baselines' / 'out',
    ROOT / '2_modern_segmentation' / 'out',
]

run_dirs = []
for sr in search_roots:
    if not sr.exists():
        continue
    for ckpt in sorted(sr.rglob('best_model.pt')):
        run_dirs.append(ckpt.parent)

run_dirs = sorted({p.resolve() for p in run_dirs})
if RUN_FILTER:
    run_dirs = [p for p in run_dirs if p.name in RUN_FILTER]

run_df = pd.DataFrame({'run_dir': [str(p) for p in run_dirs], 'run_name': [p.name for p in run_dirs]})
run_csv = OUT_DIR / 'discovered_runs.csv'
run_df.to_csv(run_csv, index=False)

print('Discovered runs:', len(run_dirs))
print('Saved:', run_csv)
display(run_df)


Discovered runs: 3
Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/cross_dataset_kvasir_sessile/discovered_runs.csv


,run_dir,run_name
0,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,deeplabv3plus_resnet50_baseline
1,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,unet_resnet34_baseline
2,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,segformer_b2_finetune


In [5]:
rows = []
failures = []

for run_dir in run_dirs:
    print(f'\nEvaluating: {run_dir.name}')
    try:
        row = evaluate_run_dir_on_metadata(
            run_dir=run_dir,
            metadata_df=sessile_df,
            out_dir=OUT_DIR,
            threshold=EVAL_THRESHOLD,
            batch_size=EVAL_BATCH_SIZE,
            num_workers=EVAL_NUM_WORKERS,
            image_size=EVAL_IMAGE_SIZE,
            split_name='external_test',
            compute_hd95=COMPUTE_HD95,
            allow_hf_download=ALLOW_HF_DOWNLOAD,
        )
        row['run_dir'] = str(run_dir)
        row['status'] = 'ok'
        rows.append(row)
        print('  dice_mean=', row.get('dice_mean'))
    except Exception as e:
        failures.append({'run_name': run_dir.name, 'run_dir': str(run_dir), 'error': str(e)})
        print('  FAILED:', e)

results_df = pd.DataFrame(rows)
if not results_df.empty:
    results_df = results_df.sort_values('dice_mean', ascending=False).reset_index(drop=True)

fail_df = pd.DataFrame(failures)

summary_csv = OUT_DIR / 'evaluation_summary.csv'
leaderboard_csv = OUT_DIR / 'leaderboard.csv'
fail_csv = OUT_DIR / 'evaluation_failures.csv'

results_df.to_csv(summary_csv, index=False)
results_df.to_csv(leaderboard_csv, index=False)
fail_df.to_csv(fail_csv, index=False)

print('Saved:', summary_csv)
print('Saved:', leaderboard_csv)
print('Saved:', fail_csv)

display(results_df)
if not fail_df.empty:
    display(fail_df)



Evaluating: deeplabv3plus_resnet50_baseline
  dice_mean= 0.47604273583172213

Evaluating: unet_resnet34_baseline
  dice_mean= 0.32789273753920434

Evaluating: segformer_b2_finetune


2026-02-15 10:49:42.921000: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


  dice_mean= 0.2868445540179004
Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/cross_dataset_kvasir_sessile/evaluation_summary.csv
Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/cross_dataset_kvasir_sessile/leaderboard.csv
Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/cross_dataset_kvasir_sessile/evaluation_failures.csv


,run_name,model_name,per_image_path,metrics_path,n,dice_mean,dice_median,dice_std,iou_mean,iou_median,...,recall_std,f1_mean,f1_median,f1_std,specificity_mean,specificity_median,specificity_std,loss,run_dir,status
0,deeplabv3plus_resnet50_baseline,deeplabv3_resnet50,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,196,0.476043,0.504818,0.289199,0.360791,0.337657,...,0.356358,0.476043,0.504818,0.289199,0.949328,0.966864,0.050108,0.405506,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,ok
1,unet_resnet34_baseline,unet_small,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,196,0.327893,0.323851,0.227390,0.219790,0.193228,...,0.344822,0.327893,0.323851,0.227390,0.895388,0.920995,0.095542,0.503712,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,ok
2,segformer_b2_finetune,segformer_binary,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,196,0.286845,0.237631,0.201058,0.185839,0.134837,...,0.300648,0.286845,0.237631,0.201058,0.791486,0.781833,0.063643,0.583483,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,ok


In [6]:
gap_rows = []
for row in rows:
    run_dir = Path(row['run_dir'])
    in_domain_path = run_dir / 'metrics_test.json'
    if not in_domain_path.exists():
        continue

    payload = json.loads(in_domain_path.read_text())
    summary = payload.get('summary', {}) if isinstance(payload, dict) else {}

    in_dice = summary.get('dice_mean')
    in_iou = summary.get('iou_mean')
    ex_dice = row.get('dice_mean')
    ex_iou = row.get('iou_mean')

    gap_rows.append({
        'run_name': row['run_name'],
        'in_domain_dice_mean': in_dice,
        'external_dice_mean': ex_dice,
        'dice_drop': (in_dice - ex_dice) if (in_dice is not None and ex_dice is not None) else None,
        'in_domain_iou_mean': in_iou,
        'external_iou_mean': ex_iou,
        'iou_drop': (in_iou - ex_iou) if (in_iou is not None and ex_iou is not None) else None,
    })

gap_df = pd.DataFrame(gap_rows)
if not gap_df.empty:
    gap_df = gap_df.sort_values('dice_drop', ascending=False).reset_index(drop=True)

gap_csv = OUT_DIR / 'transfer_gap_summary.csv'
gap_df.to_csv(gap_csv, index=False)

status = {
    'n_runs_discovered': len(run_dirs),
    'n_runs_succeeded': len(rows),
    'n_runs_failed': len(failures),
    'metadata_csv': str(meta_csv),
    'summary_csv': str(summary_csv),
    'transfer_gap_csv': str(gap_csv),
}
status_path = OUT_DIR / 'status.json'
status_path.write_text(json.dumps(status, indent=2))

print('Saved:', gap_csv)
print('Saved:', status_path)
print(json.dumps(status, indent=2))
display(gap_df)


Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/cross_dataset_kvasir_sessile/transfer_gap_summary.csv
Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/cross_dataset_kvasir_sessile/status.json
{
  "n_runs_discovered": 3,
  "n_runs_succeeded": 3,
  "n_runs_failed": 0,
  "metadata_csv": "/mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/cross_dataset_kvasir_sessile/sessile_metadata_external_test.csv",
  "summary_csv": "/mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/cross_dataset_kvasir_sessile/evaluation_summary.csv",
  "transfer_gap_csv": "/mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/cross_dataset_kvasir_sessile/transfer_gap_summary.csv"
}


,run_name,in_domain_dice_mean,external_dice_mean,dice_drop,in_domain_iou_mean,external_iou_mean,iou_drop
0,unet_resnet34_baseline,0.570666,0.327893,0.242774,0.438776,0.219790,0.218986
1,deeplabv3plus_resnet50_baseline,0.718198,0.476043,0.242156,0.611093,0.360791,0.250302
2,segformer_b2_finetune,0.474886,0.286845,0.188042,0.345191,0.185839,0.159353
